In [19]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
from astropy.table import vstack
from astroquery.gaia import Gaia
import tqdm
from astropy.table import join
from scipy.optimize import root_scalar
from astroquery.simbad import Simbad
from ipyaladin import Aladin
from astroquery.casda import Casda
from astropy.coordinates import SkyCoord
from astroquery.utils.tap.core import TapPlus
import astropy.units as u
import anywidget
import ipywidgets
import ipyaladin
import os
import re
import pandas as pd
from pathlib import Path
from astropy.io import fits
from astropy.table import join

In [17]:
sources = Table.read('data/source_obs_data.csv')
srsc_sources = Table.read('data/2026.01.16_12_07_51_SRSC_Static.csv')

First, we want a list of all the possible names of the sources we have.

In [23]:
sources_all_names = srsc_sources[np.isin(srsc_sources['Simbad_ID'], sources['Simbad_ID'])]

In [24]:
sources_all_names

Identifier,Simbad_ID,GaiaDR3_ID,TYCHO_ID,2MASS_ID,GCVS_ID,HIP_ID,UCAC4_ID,Archival_Survey,Archival_ID,Archival_epoch,Archival_ra,Archival_ra_err,Archival_dec,Archival_dec_err,Archival_ra_hms,Archival_dec_dms,Archival_parallax,Archival_parallax_err,Archival_pmra,Archival_pmra_err,Archival_pmdec,Archival_pmdec_err,Radio_multiple,Search_method_bin,1RXS_ID,TIC_ID,WISE_ID,AllWISE_ID,CNS5_ID,Stripped_Identifier,Stripped_Archival_ID
str10,str31,str28,str15,str23,str14,str10,str16,str39,str28,str19,float64,float64,float64,float64,str17,str19,float64,float64,float64,float64,float64,float64,float64,int64,str21,str14,str25,str25,str9,str9,str26
SRSC 00006,HD 102982,Gaia DR3 5368777294028600448,TYC 8228-2739-1,2MASS J11510909-5152323,--,--,UCAC4 191-064625,GaiaDR3,Gaia DR3 5368777294028600448,J2016,177.787504285457,1e-05,-51.876071495961,1e-05,11:51:09,-51:52:33.85738546,18.8,0.02,-57.34,0.02,-83.67,0.01,0.0,18,1RXS J115108.5-515223,TIC 94637805,--,--,--,SRSC00006,GaiaDR35368777294028600448
SRSC 00036,V* V376 Cep,Gaia DR3 2299477768899190400,TYC 4649-3690-1,2MASS J21581966+8252158,V* V376 Cep,HIP 108461,UCAC4 865-010582,GaiaDR3,Gaia DR3 2299477768899190400,J2016,329.577295376914,2e-05,82.870902021925,1e-05,21:58:19,+82:52:15.24727893,25.05,0.02,-137.33,0.02,-37.23,0.02,0.0,2,1RXS J215821.0+825218,TIC 470122267,--,--,--,SRSC00036,GaiaDR32299477768899190400
SRSC 00145,V* V815 Her,Gaia DR3 4590455518346609792,TYC 2103-705-1,2MASS J18081602+2941281,V* V815 Her,HIP 88848,UCAC4 599-061058,GaiaDR3,Gaia DR3 4590455518346609792,J2016,272.06734740577,9e-05,29.6910259723,0.0001,18:08:16.16337739,+29:41:27.6935003,31.2,0.1,92.5,0.1,-35.5,0.1,0.0,2,1RXS J180815.9+294127,TIC 320959269,--,--,--,SRSC00145,GaiaDR34590455518346609792
SRSC 00175,HD 165590A,--,TYC 1566-747-1,--,V* V772 Her,HIP 88637A,--,TYCHO,TYC 1566-747-1,J2000,271.457229,--,21.44599,--,18:05:49.7350,+21:26:45.56,--,--,-29.2,--,-40.3,--,2.0,2,--,--,--,--,--,SRSC00175,TYC1566-747-1
SRSC 00248,HD 82533,Gaia DR3 5312052698887995392,TYC 8584-268-1,2MASS J09304256-5331182,--,--,UCAC4 183-037492,GaiaDR3,Gaia DR3 5312052698887995392,J2016,142.67769359508,1e-05,-53.521472152447,1e-05,09:30:43,-53:31:17.29974881,18.37,0.02,43.94,0.02,52.09,0.02,0.0,18,1RXS J093042.6-533120,TIC 298079609,--,--,--,SRSC00248,GaiaDR35312052698887995392
SRSC 00255,HD 217344,Gaia DR3 6554844401276275712,TYC 7509-679-1,2MASS J23002815-3344422,V* TZ PsA,HIP 113598,UCAC4 282-218992,GaiaDR3,Gaia DR3 6554844401276275712,J2016,345.11766870906,2e-05,-33.745748964489,2e-05,23:00:28,-33:44:44.69627216,11.93,0.03,61.64,0.02,-154.0,0.02,0.0,2,1RXS J230028.2-334438,TIC 229066844,--,--,--,SRSC00255,GaiaDR36554844401276275712
SRSC 00267,HD 8558,Gaia DR3 4909846500703006976,TYC 8477-5-1,2MASS J01232126-5728507,--,HIP 6485,UCAC4 163-001271,GaiaDR3,Gaia DR3 4909846500703006976,J2016,20.839329303534,1e-05,-57.480907517274,1e-05,01:23:21,-57:28:51.26706218,22.09,0.02,92.85,0.02,-36.14,0.02,0.0,2,1RXS J012320.9-572853,TIC 206592394,--,--,--,SRSC00267,GaiaDR34909846500703006976
SRSC 00601,HD 200244,Gaia DR3 6807464138426082816,TYC 6923-323-1,2MASS J21025831-2256008,--,--,UCAC4 336-203460,GaiaDR3,Gaia DR3 6807464138426082816,J2016,315.742708658278,2e-05,-22.933914165698,1e-05,21:02:58,-22:56:02.09099651,7.17,0.02,-48.27,0.02,-62.09,0.02,0.0,2,1RXS J210258.7-225603,TIC 444149024,--,--,--,SRSC00601,GaiaDR36807464138426082816
SRSC 00628,HD 162186,Gaia DR3 4043524762244704000,TYC 7382-20-1,2MASS J17510684-3218276,--,HIP 87368,UCAC4 289-131340,GaiaDR3,Gaia DR3 4043524762244704000,J2016,267.778490987213,2e-05,-32.307805803789,1e-05,17:51:07,-32:18:28.10089364,7.43,0.02,-6.68,0.02,-27.99,0.02,0.0,2,--,TIC 444656050,--,--,--,SRSC00628,GaiaDR34043524762244704000


Now we can read in our flare catalog and see if there are any matches

In [27]:
tap = TapPlus(url="https://tapvizier.cds.unistra.fr/TAPVizieR/tap")

query = '''
SELECT TIC, SimbadName
FROM "J/AJ/159/60/table2"
'''

job = tap.launch_job(query)   # sync, not async
flare_cat = job.get_results()

In [28]:
flare_cat

TIC,SimbadName
int32,object
160113240,2MASS J23581533-3935341
183984201,2MASS J23545885-3815088
120442975,2MASS J00024007-3647548
120522722,2MASS J00111532-3756553
120488589,2MASS J00064492-3633369
70798821,2MASS J00084783-3519577
224278523,2MASS J23415428-3558397
70761819,2MASS J00025052-3416201
77041173,2MASS J23540928-3316266


And lastly, check if there are any matches in our TIC IDs

In [29]:
tic_ids = [f'TIC {ticid}' for ticid in flare_cat['TIC']]

In [31]:
flare_matches = sources_all_names[np.isin(sources_all_names['TIC_ID'], tic_ids)]

In [32]:
flare_matches

Identifier,Simbad_ID,GaiaDR3_ID,TYCHO_ID,2MASS_ID,GCVS_ID,HIP_ID,UCAC4_ID,Archival_Survey,Archival_ID,Archival_epoch,Archival_ra,Archival_ra_err,Archival_dec,Archival_dec_err,Archival_ra_hms,Archival_dec_dms,Archival_parallax,Archival_parallax_err,Archival_pmra,Archival_pmra_err,Archival_pmdec,Archival_pmdec_err,Radio_multiple,Search_method_bin,1RXS_ID,TIC_ID,WISE_ID,AllWISE_ID,CNS5_ID,Stripped_Identifier,Stripped_Archival_ID
str10,str31,str28,str15,str23,str14,str10,str16,str39,str28,str19,float64,float64,float64,float64,str17,str19,float64,float64,float64,float64,float64,float64,float64,int64,str21,str14,str25,str25,str9,str9,str26
SRSC 00255,HD 217344,Gaia DR3 6554844401276275712,TYC 7509-679-1,2MASS J23002815-3344422,V* TZ PsA,HIP 113598,UCAC4 282-218992,GaiaDR3,Gaia DR3 6554844401276275712,J2016,345.11766870906,2e-05,-33.745748964489,2e-05,23:00:28,-33:44:44.69627216,11.93,0.03,61.64,0.02,-154.0,0.02,0.0,2,1RXS J230028.2-334438,TIC 229066844,--,--,--,SRSC00255,GaiaDR36554844401276275712
SRSC 00267,HD 8558,Gaia DR3 4909846500703006976,TYC 8477-5-1,2MASS J01232126-5728507,--,HIP 6485,UCAC4 163-001271,GaiaDR3,Gaia DR3 4909846500703006976,J2016,20.839329303534,1e-05,-57.480907517274,1e-05,01:23:21,-57:28:51.26706218,22.09,0.02,92.85,0.02,-36.14,0.02,0.0,2,1RXS J012320.9-572853,TIC 206592394,--,--,--,SRSC00267,GaiaDR34909846500703006976
SRSC 00822,HD 13183,Gaia DR3 4743944043046490240,TYC 8483-917-1,2MASS J02071805-5311565,--,HIP 9892,UCAC4 185-002009,GaiaDR3,Gaia DR3 4743944043046490240,J2016,31.825892379213,1e-05,-53.199137776857,1e-05,02:07:18,-53:11:56.89599669,20.16,0.02,86.52,0.02,-22.79,0.02,0.0,2,1RXS J020718.6-531155,TIC 219998026,--,--,--,SRSC00822,GaiaDR34743944043046490240
SRSC 01317,TYC 9143-577-1,Gaia DR3 4697476998028774016,TYC 9143-577-1,2MASS J01530751-6757141,--,--,--,GaiaDR3,Gaia DR3 4697476998028774016,J2016.0,28.282259510166043,1.05771049857139e-05,-67.95397668077126,8.9625688269734e-06,1:53:07.74228244,-67:57:14.31605078,8.496029074638077,0.0105876754969358,81.09031197909817,0.0140872970223426,-11.162374336794835,0.0118996948003768,--,2,--,TIC 232073646,--,--,--,SRSC01317,GaiaDR34697476998028774016
SRSC 01755,HD 205297,Gaia DR3 6402182228283074944,TYC 9111-182-1,2MASS J21371556-6502196,--,--,--,GaiaDR3,Gaia DR3 6402182228283074944,J2016.0,324.31493765302343,8.415186e-06,-65.03918738505827,1.124421e-05,21:37:15.58503673,-65:02:21.07458621,13.609369855997269,0.013667652,7.873563341134037,0.011004537,-99.94544130406511,0.01468092,--,2,1RXS J213714.3-650210,TIC 403121294,--,--,--,SRSC01755,GaiaDR36402182228283074944
SRSC 01761,HD 202917,Gaia DR3 6463782435948431616,TYC 8806-401-1,2MASS J21204994-5302030,--,HIP 105388,--,GaiaDR3,Gaia DR3 6463782435948431616,J2016.0,320.2083890093772,1.03829866275191e-05,-53.0346347389645,1.16160763427615e-05,21:20:50.01336225,-53:02:04.68506027,21.409293944288,0.0157449971884489,31.42732327368997,0.012646242044866,-95.58701420119108,0.0138090215623378,--,2,1RXS J212050.3-530200,TIC 79403675,WISE J212049.98-530204.0,WISEA J212049.99-530204.2,--,SRSC01761,GaiaDR36463782435948431616
